# Análise de E-commerce (Olist)
Dataset público de ~100 mil pedidos da Olist (2016–2018). Perguntas: como evoluíram as vendas, o que e onde se vende, como é a entrega e quanto ela afeta a satisfação.

## 1. Carga e exploração dos dados

In [ ]:
import pandas as pd

pasta = "CSV/"

orders = pd.read_csv(pasta + "olist_orders_dataset.csv")
items = pd.read_csv(pasta + "olist_order_items_dataset.csv")
payments = pd.read_csv(pasta + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(pasta + "olist_order_reviews_dataset.csv")
customers = pd.read_csv(pasta + "olist_customers_dataset.csv")
sellers = pd.read_csv(pasta + "olist_sellers_dataset.csv")
products = pd.read_csv(pasta + "olist_products_dataset.csv")
categorias = pd.read_csv(pasta + "product_category_name_translation.csv")

In [ ]:
orders.shape

In [ ]:
orders.info()

In [ ]:
colunas_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in colunas_data:
    orders[col] = pd.to_datetime(orders[col])

In [ ]:
orders.isnull().sum()

In [ ]:
orders["order_status"].value_counts()

In [ ]:
entregues = orders[orders["order_status"] == "delivered"].copy()

In [ ]:
entregues["dias_entrega"] = (
    entregues["order_delivered_customer_date"] - entregues["order_purchase_timestamp"]
).dt.days

In [ ]:
entregues["dias_entrega"].describe()

## 4. Entregas: quantos pedidos chegam atrasados?

In [ ]:
entregues["atrasou"] = (
    entregues["order_delivered_customer_date"] > entregues["order_estimated_delivery_date"]
)

pct_atraso = entregues["atrasou"].mean() * 100
print(f"Pedidos entregues com atraso: {pct_atraso:.1f}%")
print(f"Atraso médio (só os atrasados): {(entregues.loc[entregues['atrasou'], 'order_delivered_customer_date'] - entregues.loc[entregues['atrasou'], 'order_estimated_delivery_date']).dt.days.mean():.1f} dias")

## 5. Juntando as tabelas
Uma linha por item vendido, já com data do pedido, cliente, categoria (em inglês) e estado.

In [ ]:
vendas = (
    items
    .merge(orders[["order_id", "customer_id", "order_status", "order_purchase_timestamp"]], on="order_id")
    .merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id")
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(categorias, on="product_category_name", how="left")
)

# só pedidos entregues entram na análise de receita
vendas = vendas[vendas["order_status"] == "delivered"].copy()
vendas["mes"] = vendas["order_purchase_timestamp"].dt.to_period("M")

print(vendas.shape)
vendas.head()

## 6. Receita mensal
Os meses de borda (início da base em 2016 e o último mês em 2018) têm poucos pedidos, então ficam de fora do gráfico para não distorcer a tendência.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")
os.makedirs("imagens", exist_ok=True)

receita_mes = vendas.groupby("mes")["price"].sum()
pedidos_mes = vendas.groupby("mes")["order_id"].nunique()
print(pedidos_mes.head(4), pedidos_mes.tail(3), sep="\n\n")

In [ ]:
receita_plot = receita_mes.loc["2017-01":"2018-08"]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(receita_plot.index.astype(str), receita_plot.values / 1e3, marker="o", color="#2a6fdb")
ax.set_title("Receita mensal (pedidos entregues)")
ax.set_ylabel("R$ mil")
ax.set_xlabel("")
plt.xticks(rotation=60)
plt.tight_layout()
plt.savefig("imagens/receita_mensal.png", dpi=150)
plt.show()

cresc = (receita_plot["2018-01":"2018-08"].mean() / receita_plot["2017-01":"2017-08"].mean() - 1) * 100
print(f"Média mensal jan–ago/2018 vs jan–ago/2017: {cresc:+.0f}%")

## 7. Categorias que mais vendem

In [ ]:
top_cat = (
    vendas.groupby("product_category_name_english")
    .agg(receita=("price", "sum"), itens=("order_id", "count"))
    .sort_values("receita", ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top_cat["receita"] / 1e3, y=top_cat.index, color="#2a6fdb", ax=ax)
ax.set_title("Top 10 categorias por receita")
ax.set_xlabel("R$ mil")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("imagens/top_categorias.png", dpi=150)
plt.show()

top_cat.assign(participacao_pct=(top_cat["receita"] / vendas["price"].sum() * 100).round(1))

## 8. Onde estão os clientes: receita e entrega por estado

In [ ]:
por_estado = (
    vendas.groupby("customer_state")
    .agg(receita=("price", "sum"), pedidos=("order_id", "nunique"))
    .sort_values("receita", ascending=False)
)
por_estado["participacao_pct"] = (por_estado["receita"] / por_estado["receita"].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=por_estado.head(10)["receita"] / 1e3, y=por_estado.head(10).index, color="#2a6fdb", ax=ax)
ax.set_title("Top 10 estados por receita")
ax.set_xlabel("R$ mil")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("imagens/receita_estado.png", dpi=150)
plt.show()

por_estado.head(10)

In [ ]:
entrega_estado = (
    entregues.merge(customers[["customer_id", "customer_state"]], on="customer_id")
    .groupby("customer_state")["dias_entrega"]
    .agg(["mean", "count"])
    .query("count >= 100")
    .sort_values("mean")
)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=entrega_estado["mean"], y=entrega_estado.index, color="#e07b39", ax=ax)
ax.set_title("Prazo médio de entrega por estado (dias)")
ax.set_xlabel("dias")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("imagens/entrega_estado.png", dpi=150)
plt.show()

entrega_estado.round(1).iloc[[0, 1, 2, -3, -2, -1]]

## 9. Formas de pagamento

In [ ]:
pagamento = payments.groupby("payment_type").agg(
    pedidos=("order_id", "nunique"), valor=("payment_value", "sum")
)
pagamento["pct_pedidos"] = (pagamento["pedidos"] / pagamento["pedidos"].sum() * 100).round(1)
pagamento = pagamento.sort_values("pedidos", ascending=False)

cartao = payments[payments["payment_type"] == "credit_card"]
print(f"Cartão de crédito parcelado (2x ou mais): {(cartao['payment_installments'] > 1).mean() * 100:.1f}% dos pagamentos")
pagamento

## 10. Avaliações dos clientes
Um pedido pode ter mais de uma avaliação, então fica a mais recente por pedido.

In [ ]:
avaliacoes = (
    reviews.sort_values("review_creation_date")
    .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]]
)

print(f"Nota média: {avaliacoes['review_score'].mean():.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x="review_score", data=avaliacoes, color="#2a6fdb", ax=ax)
ax.set_title("Distribuição das notas")
ax.set_xlabel("nota")
ax.set_ylabel("pedidos")
plt.tight_layout()
plt.savefig("imagens/notas.png", dpi=150)
plt.show()

(avaliacoes["review_score"].value_counts(normalize=True).sort_index() * 100).round(1)

## 11. Atraso na entrega derruba a nota?

In [ ]:
entrega_nota = entregues.merge(avaliacoes, on="order_id")

nota_por_atraso = entrega_nota.groupby("atrasou")["review_score"].mean()
print(nota_por_atraso.round(2))

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=["No prazo", "Atrasado"], y=nota_por_atraso.values, hue=["No prazo", "Atrasado"],
            palette=["#2a6fdb", "#d94f4f"], legend=False, ax=ax)
ax.set_ylim(0, 5)
ax.set_title("Nota média: no prazo vs atrasado")
ax.set_ylabel("nota média")
plt.tight_layout()
plt.savefig("imagens/nota_atraso.png", dpi=150)
plt.show()

nota1_2 = entrega_nota.groupby("atrasou")["review_score"].apply(lambda s: (s <= 2).mean() * 100).round(1)
nota1_2

## 12. Clientes que voltam a comprar

In [ ]:
pedidos_cliente = (
    orders[orders["order_status"] == "delivered"]
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
    .groupby("customer_unique_id")["order_id"].nunique()
)

recorrentes = (pedidos_cliente > 1).mean() * 100
print(f"Clientes únicos: {len(pedidos_cliente):,}")
print(f"Compraram mais de uma vez: {recorrentes:.1f}%")

ticket = vendas.groupby("order_id")["price"].sum().mean()
print(f"Ticket médio (só produtos): R$ {ticket:.2f}")

## 13. Conclusões

- **Crescimento forte:** a receita mensal média de jan–ago/2018 foi cerca de 141% maior que a de jan–ago/2017.
- **Pico em nov/2017** (provável Black Friday), seguido de queda em dezembro.
- **Concentração geográfica:** SP responde por 38,3% da receita; SP, RJ e MG juntos passam de 63%.
- **Categorias:** `health_beauty`, `watches_gifts` e `bed_bath_table` lideram em receita (9,3%, 8,8% e 7,7%). `bed_bath_table` vende mais itens, mas com ticket menor.
- **Entrega:** prazo médio de 12 dias, e 8,1% dos pedidos chegam depois da data estimada. SP recebe em ~8 dias; estados do Norte (PA, AM) passam de 23 dias.
- **Atraso pesa na satisfação:** a nota média cai de 4,29 (no prazo) para 2,57 (atrasado), e 54,1% dos pedidos atrasados recebem nota 1 ou 2, contra 9,2% dos entregues no prazo.
- **Pagamento:** 75% dos pedidos usam cartão de crédito, e 67% desses pagamentos são parcelados; boleto vem em seguida com ~20%.
- **Retenção baixa:** só 3,0% dos clientes compraram mais de uma vez, então há espaço para ações de recompra.

**Limitações:** a base cobre 2016–2018 e os meses de borda têm poucos pedidos; a receita considera só o preço dos itens (sem frete) e só pedidos entregues.
